# Model Evaluation: YOLOv8n + Position-Aware CBAM (PA-CBAM)
**Experiment ID:** `pacbam`  
**Architecture:** Position-Aware CBAM (PA-CBAM)  
**Dataset:** Helmet Detection (2 classes: helmet, non-helmet)  
> Run this notebook to evaluate.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
EXPERIMENT_ID = "pacbam"
WEIGHTS       = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/models/best (PACBAM).pt"
DATA_YAML     = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/data_big.yaml"
DATASET_ROOT  = r"C:\Users\soumy\OneDrive\Desktop\test(big)"
IMGSZ         = 640
BATCH         = 16
CONF_THRES    = 0.001
IOU_THRES     = 0.5
OUTPUT_DIR    = r"C:\Users\soumy\OneDrive\Desktop\test(big)/../Testing_Scripts(final)/eval_small_pacbam"
CLASS_NAMES   = ["helmet", "non-helmet"]

In [ ]:
!pip install ultralytics -q

In [ ]:
import torch, torch.nn as nn, sys

class AddCoords(nn.Module):
    def __init__(self, with_r=False):
        super().__init__()
        self.with_r = with_r
    def forward(self, x):
        b, c, h, w = x.shape
        device, dtype = x.device, x.dtype
        y_coords = torch.linspace(-1, 1, h, device=device, dtype=dtype)
        x_coords = torch.linspace(-1, 1, w, device=device, dtype=dtype)
        yy, xx = torch.meshgrid(y_coords, x_coords, indexing='ij')
        xx = xx.unsqueeze(0).unsqueeze(0).expand(b, 1, h, w)
        yy = yy.unsqueeze(0).unsqueeze(0).expand(b, 1, h, w)
        coords = [x, xx, yy]
        if self.with_r: coords.append(torch.sqrt(xx**2 + yy**2))
        return torch.cat(coords, dim=1)

class ChannelAttentionPA(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(channels, mid, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(mid, channels, 1, bias=False))
    def forward(self, x):
        return x * torch.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttentionPA(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(4, 1, kernel_size, padding=kernel_size//2, bias=False)
    def forward(self, x, coords):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg_out, max_out, coords], dim=1)))

class PACBAM(nn.Module):
    def __init__(self, kernel_size=7, reduction=16):
        super().__init__()
        self.kernel_size = kernel_size
        self.reduction = reduction
        self.channel_attention = None
        self.spatial_attention = SpatialAttentionPA(kernel_size)
        self.add_coords = AddCoords(with_r=False)
    def forward(self, x):
        if self.channel_attention is None:
            self.channel_attention = ChannelAttentionPA(x.shape[1], self.reduction).to(x.device)
        x = self.channel_attention(x)
        dummy = torch.zeros(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype)
        coords = self.add_coords(dummy)[:, 1:3, :, :]
        x = self.spatial_attention(x, coords)
        return x

import ultralytics.nn.tasks as tasks, ultralytics.nn.modules as modules
for cls in [AddCoords, ChannelAttentionPA, SpatialAttentionPA, PACBAM]:
    setattr(tasks, cls.__name__, cls)
    setattr(modules, cls.__name__, cls)
    for s in ("block","conv","head"):
        if hasattr(modules,s): setattr(getattr(modules,s), cls.__name__, cls)
    sys.modules['__main__'].__dict__[cls.__name__] = cls
torch.serialization.add_safe_globals([AddCoords, ChannelAttentionPA, SpatialAttentionPA, PACBAM])
print("[OK] PACBAM registered")

In [ ]:
import os, json, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[OK] Output directory: {OUTPUT_DIR}")
print(f"[OK] Loading weights : {WEIGHTS}")
model = YOLO(WEIGHTS)


## 1 - Validation on Test Split

In [ ]:
test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    conf=CONF_THRES,
    iou=IOU_THRES,
    save_json=True,
    plots=True,
    project=OUTPUT_DIR,
    name="test",
    exist_ok=True,
    verbose=True,
)

print("\n[Test] Raw metrics object:", test_results)

## 2 - Extract and Display Metrics

In [ ]:
def extract_metrics(results, split_name):
    box = results.box
    metrics = {
        "experiment": EXPERIMENT_ID, "split": split_name,
        "mAP50": round(float(box.map50), 4), "mAP50_95": round(float(box.map), 4),
        "precision": round(float(box.mp), 4), "recall": round(float(box.mr), 4),
        "f1": round(float(np.mean(box.f1)), 4),
    }
    for i, cls_name in enumerate(CLASS_NAMES):
        try:
            metrics[f"AP50_{cls_name}"] = round(float(box.ap50[i]), 4)
            metrics[f"AP50_95_{cls_name}"] = round(float(box.ap[i]), 4)
            metrics[f"P_{cls_name}"] = round(float(box.p[i]), 4)
            metrics[f"R_{cls_name}"] = round(float(box.r[i]), 4)
            metrics[f"F1_{cls_name}"] = round(float(box.f1[i]), 4)
        except Exception: pass
    return metrics

test_metrics = extract_metrics(test_results, "test")
fitness = round(float(test_results.fitness), 4)
speed = test_results.speed
print("=" * 55)
print(f"  EXPERIMENT : {EXPERIMENT_ID.upper()}")
print("=" * 55)
print(f"  mAP@50      : {test_metrics['mAP50']}")
print(f"  mAP@50-95   : {test_metrics['mAP50_95']}")
print(f"  Precision   : {test_metrics['precision']}")
print(f"  Recall      : {test_metrics['recall']}")
print(f"  F1          : {test_metrics['f1']}")
print(f"  Fitness     : {fitness}")
print(f"  Inference   : {speed['inference']:.2f} ms/img")
print(f"  Preprocess  : {speed['preprocess']:.2f} ms/img")
print(f"  Postprocess : {speed['postprocess']:.2f} ms/img")
for cls in CLASS_NAMES:
    print(f"  AP50 [{cls:<12}]: {test_metrics.get(f'AP50_{cls}', 'N/A')}")
print("=" * 55)


## 3 - Save Metrics

In [ ]:
test_metrics["fitness"] = fitness
test_metrics["inference_ms_img"] = round(speed["inference"], 2)
test_metrics["preprocess_ms_img"] = round(speed["preprocess"], 2)
test_metrics["postprocess_ms_img"] = round(speed["postprocess"], 2)

csv_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.csv"
df = pd.DataFrame([test_metrics])
df.to_csv(csv_path, index=False)
print(f"[OK] CSV saved -> {csv_path}")

json_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.json"
with open(json_path, "w") as f:
    json.dump([test_metrics], f, indent=2)
print(f"[OK] JSON saved -> {json_path}")

std_csv = f"{OUTPUT_DIR}/test_results.csv"
std_row = {"Model Name": EXPERIMENT_ID, "Precision": test_metrics["precision"],
           "Recall": test_metrics["recall"], "F1": test_metrics["f1"],
           "mAP50": test_metrics["mAP50"], "mAP50_95": test_metrics["mAP50_95"],
           "Fitness": fitness, "Inference_ms_img": round(speed["inference"], 2)}
pd.DataFrame([std_row]).to_csv(std_csv, index=False)
print(f"[OK] Standardized CSV -> {std_csv}")
df

## 4 - Metrics Bar Chart

In [ ]:
metric_keys = ["mAP50","mAP50_95","precision","recall","f1"]
metric_labels = ["mAP@50","mAP@50-95","Precision","Recall","F1"]
test_vals = [test_metrics[k] for k in metric_keys]
x = np.arange(len(metric_labels))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, test_vals, 0.5, color="#DD8452", alpha=0.88)
ax.set_ylim(0, 1.05); ax.set_xticks(x); ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title(f"Test Metrics - {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.5)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x()+bar.get_width()/2, h), xytext=(0,3), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout()
chart_path = f"{OUTPUT_DIR}/metrics_chart_{EXPERIMENT_ID}.png"
plt.savefig(chart_path, dpi=150); plt.show()
print(f"[OK] Chart saved -> {chart_path}")

## 5 - Per-Class AP

In [ ]:
ap50_vals = [test_metrics.get(f"AP50_{c}", 0) for c in CLASS_NAMES]
ap95_vals = [test_metrics.get(f"AP50_95_{c}", 0) for c in CLASS_NAMES]
x = np.arange(len(CLASS_NAMES)); width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x-width/2, ap50_vals, width, label="AP@50", color="#55A868", alpha=0.88)
b2 = ax.bar(x+width/2, ap95_vals, width, label="AP@50-95", color="#C44E52", alpha=0.88)
ax.set_ylim(0,1.05); ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylabel("AP Score", fontsize=12)
ax.set_title(f"Per-Class AP (Test) - {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.legend(fontsize=11); ax.grid(axis="y", linestyle="--", alpha=0.5)
for bar in [*b1,*b2]:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x()+bar.get_width()/2,h), xytext=(0,3), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout()
pc_path = f"{OUTPUT_DIR}/per_class_ap_{EXPERIMENT_ID}.png"
plt.savefig(pc_path, dpi=150); plt.show()
print(f"[OK] Per-class chart saved -> {pc_path}")

## 6 - Model Info and Final Summary

In [ ]:
info = model.info(verbose=True)
print(f"\n  Weights: {WEIGHTS}")
print("\n" + "=" * 60)
print(f"  FINAL SUMMARY - {EXPERIMENT_ID.upper()}")
print("=" * 60)
for key, label in zip(metric_keys, metric_labels):
    print(f"  {label:<20} {test_metrics[key]:>10.4f}")
print(f"  {'Fitness':<20} {fitness:>10.4f}")
print(f"  {'Inference ms/img':<20} {speed['inference']:>10.2f}")
print("=" * 60)
print(f"\n  Outputs saved to: {OUTPUT_DIR}/")
